In [ ]:
%pip install -U trl

In [ ]:
from importlib import reload

from Trainers.trainer_classifier import ClassifierTrainer, ClassifierTrainingConfig
from Trainers.trainer_ppo import PPOTrainingConfig, PolicyPPOTrainer
from Models.model_policy import PolicyModel
from Models.model_value import ValueModel
from Models.model_reward import RewardModel
from Models.model_classifier import Classifier
from Models.lora import LoRASettings
import Models.model_evaluator as model_evaluator
import Datasets.dataset_request as dataset_request
from dataclasses import dataclass
from datasets import load_dataset
import torch
from pathlib import Path

import gc


dataset_request = reload(dataset_request)
RequestDataset = dataset_request.RequestDataset
DatasetClassifier = dataset_request.DatasetClassifier

In [ ]:
dataset = RequestDataset.load("human_requests_hh-rlhf.pt", "Qwen/Qwen3-0.6B")
dataset.truncate(0, 5)

In [ ]:
config = PPOTrainingConfig(
    output_dir="outputs/ppo_policy"
)
policy = PolicyModel("Qwen/Qwen3-0.6B")
value = ValueModel("Qwen/Qwen3-0.6B")
reward_model = RewardModel("Skywork/Skywork-Reward-V2-Qwen3-0.6B", "proxy")
judge = RewardModel("Skywork/Skywork-Reward-V2-Qwen3-4B", "judge")

In [ ]:
policy.generate_new_dataset(dataset, 4)

In [ ]:
reward_model.init_normalization(policy)
judge.init_normalization(policy)

reward_model.score_policy(policy)
judge.score_policy(policy)

In [ ]:
classfier_dataset = policy.generate_dataset_classifier(2)
train, test = classfier_dataset.split()

In [ ]:
classifier_config = ClassifierTrainingConfig(output_dir="outputs/classifiers")
classifier = Classifier("1", "Qwen/Qwen3-0.6B")

classifier_trainer = ClassifierTrainer(classifier, classifier_config)
classifier_trainer.train(train, test)

In [ ]:



# 1) Create dataset
dataset_name: str = "Anthropic/hh-rlhf"
dataset = load_dataset(dataset_name)
dataset = RequestDataset.from_raw(dataset, "Qwen/Qwen3-0.6B")
dataset.truncate(config.start_dataset, config.end_dataset)


# 2) policy generates answers
if config.policy_load_path is not None:
    policy = PolicyModel.load(config.policy_load_path)

else:
    policy = PolicyModel(config.policy_name)

policy.generate_new_dataset(dataset)


# 3) Put policy in cpu
policy.offload()


# 4) score with reward and del
proxy = RewardModel(config.reward_model_name, config.reward_mode_name)
proxy.score_policy(policy)

proxy.model.to("cpu")
del proxy
gc.collect()
torch.cuda.empty_cache()



judge = RewardModel(config.judge_model_name, config.judge_mode_name)
judge.score_policy(policy)


judge.model.to("cpu")
del judge
gc.collect()
torch.cuda.empty_cache()


# 6) creates new dataset for classifier
kwargs = {}
kwargs["prompts"], kwargs["answers"], kwargs["reward_scores"], kwargs["judge_scores"] = policy.return_rows()
dataset_for_classifier = DatasetClassifier()

dataset_for_classifier.add(**kwargs)


#  -----train classifier-----

# 7) Create classifier
classifier = Classifier(config.classifier_model_name)


# 8) Split to train and test and create config -> 0.8, 0.2
train, test = dataset_for_classifier.split()
classifier_config = ClassifierTrainingConfig(
    output_dir=f"outputs/classifiers/id={classifier.id}",
    lora_settings=LoRASettings(
        rank=16,
        alpha=32,
        dropout=0.05,
        target_modules="all-linear",
    ),
)


# 9) Create trainer
classifier_trainer = ClassifierTrainer(classifier, classifier_config)


# 10) Train
hf_trainer = classifier_trainer.train(train)


# 11) Eval
classifier_trainer.evaluate(hf_trainer, test)